# 02 - Entraînement du modèle

Ce notebook implémente la boucle d'entraînement du `CurlClassifier` avec une validation croisée de type Leave-One-Out.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import sys
import os

sys.path.append('../src')
from model import CurlClassifier
from utils import load_dataset

## 1. Chargement et préparation

Nous chargeons les données et les convertissons au format Tensor PyTorch.

In [2]:
base_path = '../data'
X, y = load_dataset(base_path)

# Transpose to (batch, channels, seq_len)
X = np.transpose(X, (0, 2, 1))

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

print(f"X shape: {X_tensor.shape}")
print(f"y shape: {y_tensor.shape}")

X shape: torch.Size([65, 6, 100])
y shape: torch.Size([65])


## 2. Configuration de l'entraînement

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lr = 1e-3
batch_size = 16
num_epochs = 50

model = CurlClassifier().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

## 3. Boucle d'entraînement (Simple)

Ici nous entraînons sur tout le dataset pour obtenir le modèle final.

In [4]:
dataset = TensorDataset(X_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(loader):.4f}")

# Sauvegarde du modèle
os.makedirs('../models_saved', exist_ok=True)
torch.save(model.state_dict(), '../models_saved/curl_classifier.pth')
print("Modèle sauvegardé dans models_saved/curl_classifier.pth")

Epoch 10/50, Loss: 0.0892
Epoch 20/50, Loss: 0.0075
Epoch 30/50, Loss: 0.0032
Epoch 40/50, Loss: 0.0007
Epoch 50/50, Loss: 0.0009
Modèle sauvegardé dans models_saved/curl_classifier.pth
